# 01 — D-01 Canonical Pair Registry

이 notebook은 Claude가 freeze한 research contract와 data-recon exact ingest allowlist를 읽어 Phase-1 pair registry를 구축한다. 정규화, tokenizer, 형태소 분석, Phase-2 QC exclusion은 수행하지 않는다. 지원 모듈은 streaming parser, provenance/content identity, DuckDB group resolution, artifact persistence를 구현하며 이 notebook은 그 입력·출력·검증을 순서대로 드러낸다.

실행 계약: fresh kernel에서 위에서 아래로 실행하며 `TOKENIZATION_PREMIUM_RESEARCH_CONFIG`와 `TOKENIZATION_PREMIUM_INGEST_CONTRACT`는 agent branch 검증 시에만 외부 contract worktree를 가리킬 수 있다. 통합 후 canonical 실행에서는 repository 내부 기본 경로를 사용한다.

## Cell 01.01 — 실행 경로와 D-01 지원 계층 초기화

- Research Spec:
  - §8, §9, §11, §12.1, §30.2, §38
  - Gate G1
- 목적: 실행 worktree, canonical root, raw root, contract 및 artifact 경로를 숨김 없이 확정한다.
- 입력: 현재 working directory와 선택적 environment override.
- 전제: editable package가 현재 repository/worktree의 `src/tokenization_premium`을 가리킨다.
- 수행: package root에서 repository root를 resolve하고 project containment 및 필수 입력 존재를 assert한다.
- 출력: 실행에 사용할 모든 절대경로.
- 저장 Artifact: 없음.
- 검증: execution root에 pyproject.toml이 있고 raw/contract/notebook 파일이 존재한다.
- 실패 조건: 경로 누락, project root 오판, contract 부재.
- 다음 셀과의 관계: 검증된 contract path를 01.02에 전달한다.

In [1]:
import os  # agent/canonical 실행에서 contract 위치를 명시적으로 override하기 위해 환경변수를 읽는다.
from pathlib import Path  # project와 artifact 경로를 안전하게 구성한다.

import pandas as pd  # 작은 contract/reconciliation 표를 사람이 검토할 DataFrame으로 표현한다.
import pyarrow.parquet as pq  # 저장된 Parquet metadata와 schema를 재검증한다.
from IPython.display import display  # notebook narrative 안에서 검증 표를 명시적으로 표시한다.

from tokenization_premium.hashing import sha256_file  # contract와 artifact bytes의 project 공통 SHA-256을 계산한다.
from tokenization_premium.paths import PROJECT_ROOT as PACKAGE_PROJECT_ROOT  # kernel cwd와 무관하게 editable package가 속한 repository root를 찾는다.
from tokenization_premium.registry import build_manifest, finalize_registry, load_contracts, persist_manifest, validate_registry, verify_ingest_allowlist, write_reconciliation_csv, write_source_registry, write_staging_registry  # D-01 streaming ingest·identity·group·artifact 지원 계층을 호출한다.
from tokenization_premium.schemas import pair_registry_schema, source_registry_schema  # notebook에서 final schema 계약을 직접 검증한다.

EXECUTION_ROOT = PACKAGE_PROJECT_ROOT.resolve()  # nbconvert가 notebooks/를 cwd로 사용해도 package가 속한 실제 worktree root를 고정한다.
assert (EXECUTION_ROOT / 'pyproject.toml').is_file()  # repository root가 아닌 cwd에서의 오실행을 즉시 중단한다.
CANONICAL_ROOT = Path(os.environ.get('TOKENIZATION_PREMIUM_CANONICAL_ROOT', '/home/sieg/projects-wsl/Tokenization_Premium')).resolve()  # manifest의 canonical project identity를 agent worktree와 분리한다.
RAW_ROOT = Path(os.environ.get('TOKENIZATION_PREMIUM_RAW_ROOT', str(CANONICAL_ROOT / 'data/raw/aigub'))).resolve()  # immutable local raw root를 data-recon contract와 같은 위치로 해석한다.
RESEARCH_CONFIG_PATH = Path(os.environ.get('TOKENIZATION_PREMIUM_RESEARCH_CONFIG', str(EXECUTION_ROOT / 'configs/research_v1.yaml'))).resolve()  # 통합 후 repository 내부 Claude contract를 기본값으로 사용한다.
INGEST_CONTRACT_PATH = Path(os.environ.get('TOKENIZATION_PREMIUM_INGEST_CONTRACT', str(EXECUTION_ROOT / 'outputs/manifests/data_recon/G1_INGEST_EXPECTATIONS_v001.json'))).resolve()  # 통합 후 repository 내부 data-recon oracle을 기본값으로 사용한다.
NOTEBOOK_PATH = EXECUTION_ROOT / 'notebooks/01_build_pair_registry.ipynb'  # manifest에 입력 notebook hash를 연결한다.
STAGING_PATH = EXECUTION_ROOT / 'data/interim/PAIR_REGISTRY_v001_staging.parquet'  # group 계산 전 재생성 가능한 ignored staging 경로를 정한다.
PAIR_REGISTRY_PATH = EXECUTION_ROOT / 'data/registry/PAIR_REGISTRY_v001.parquet'  # canonical D-01 pair table 경로를 정한다.
SOURCE_REGISTRY_PATH = EXECUTION_ROOT / 'data/registry/SOURCE_REGISTRY_v001.parquet'  # family-grain source table 경로를 정한다.
RECONCILIATION_PATH = EXECUTION_ROOT / 'outputs/reports/PAIR_REGISTRY_RECONCILIATION_v001.csv'  # human-readable oracle report 경로를 정한다.
MANIFEST_PATH = EXECUTION_ROOT / 'outputs/manifests/PAIR_REGISTRY_MANIFEST_v001.json'  # provenance manifest 경로를 정한다.
RUNTIME_DIR = EXECUTION_ROOT / '.runtime/g1-duckdb'  # DuckDB spill을 project-contained ignored runtime 경로에 제한한다.
for required_path in (RAW_ROOT, RESEARCH_CONFIG_PATH, INGEST_CONTRACT_PATH, NOTEBOOK_PATH):  # 모든 필수 read input을 순회한다.
    assert required_path.exists(), f'필수 입력 누락: {required_path}'  # 누락 입력이 있으면 ingest 전에 fail-fast한다.
display({'execution_root': str(EXECUTION_ROOT), 'canonical_root': str(CANONICAL_ROOT), 'raw_root': str(RAW_ROOT), 'research_config': str(RESEARCH_CONFIG_PATH), 'ingest_contract': str(INGEST_CONTRACT_PATH)})  # 실제 실행 경로를 notebook evidence로 표시한다.

{'execution_root': '/home/sieg/projects-wsl/Tokenization_Premium/.agent_worktrees/codex-g1',
 'canonical_root': '/home/sieg/projects-wsl/Tokenization_Premium',
 'raw_root': '/home/sieg/projects-wsl/Tokenization_Premium/data/raw/aigub',
 'research_config': '/home/sieg/projects-wsl/Tokenization_Premium/.agent_worktrees/claude-g1-canonical/configs/research_v1.yaml',
 'ingest_contract': '/home/sieg/projects-wsl/Tokenization_Premium/.agent_worktrees/data-g1-contract/outputs/manifests/data_recon/G1_INGEST_EXPECTATIONS_v001.json'}

## Cell 01.02 — Research/Data contract strict load

- Research Spec:
  - §9.2, §9.3, §12.1
  - D-RD-05, D-RD-06, D-RD-07
  - Gate G1
- 목적: authoritative research config와 ingest oracle이 implementation-ready exact bundle인지 확인한다.
- 입력: `research_v1.yaml`, `G1_INGEST_EXPECTATIONS_v001.json`.
- 전제: Claude/data-recon 소유 파일은 read-only다.
- 수행: YAML/JSON parse 후 Tier, Legacy direction, Phase-1 status, version, expected counts를 assert한다.
- 출력: `research`, `ingest`, contract SHA-256.
- 저장 Artifact: manifest에 contract hash가 후속 저장된다.
- 검증: expected total=5,652,925; 025/026 Tier A; Legacy Tier null; Legacy direction UNKNOWN; Phase1 review.
- 실패 조건: key 누락 또는 frozen value 불일치.
- 다음 셀과의 관계: verified contracts만 exact allowlist hash audit에 전달한다.

In [2]:
research, ingest = load_contracts(RESEARCH_CONFIG_PATH, INGEST_CONTRACT_PATH)  # 두 authoritative contract를 parse하고 exact G1 bundle을 fail-fast 검증한다.
RESEARCH_CONFIG_SHA256 = sha256_file(RESEARCH_CONFIG_PATH)  # 실행이 소비한 research config bytes를 고정한다.
INGEST_CONTRACT_SHA256 = sha256_file(INGEST_CONTRACT_PATH)  # 실행이 소비한 ingest oracle bytes를 고정한다.
RAW_MANIFEST_SHA256 = str(ingest['raw_manifest_sha'])  # source identity와 lineage에 사용할 verified raw snapshot hash를 읽는다.
assert len(RAW_MANIFEST_SHA256) == 64  # raw manifest SHA-256 형식을 확인한다.
display({'research_config_sha256': RESEARCH_CONFIG_SHA256, 'ingest_contract_sha256': INGEST_CONTRACT_SHA256, 'raw_manifest_sha256': RAW_MANIFEST_SHA256, 'expected_rows': ingest['expected_logical_record_counts']})  # contract evidence를 notebook에 표시한다.

{'research_config_sha256': '1b7228351395d66f26d5da45d0ff61bf3630c9b40ca1af90e87c410f1206c8b6',
 'ingest_contract_sha256': '88261ed2b8ea4e32571813d41479104c2d2813ead9de2cacf7d0de9f51256c9e',
 'raw_manifest_sha256': '9a546bc91225e5331d0e8e48a1e06685cb5304ed7098aff5cbf40c74022c1f0c',
 'expected_rows': {'025': 2700345,
  '026': 1350162,
  'LEGACY': 1602418,
  'total': 5652925}}

## Cell 01.03 — Exact ingest allowlist와 raw SHA audit

- Research Spec:
  - §11, §12.1, §30.2
  - Gate G1
- 목적: recursive discovery 없이 6 JSON+10 XLSX canonical physical files만 ingest하도록 고정한다.
- 입력: data-recon `double_ingest_prevention_contract.ingest_allowlist`, immutable raw bytes.
- 전제: archive와 원천/라벨 alias는 ingest 대상이 아니다.
- 수행: path uniqueness, file existence, 실제 SHA-256, family record-count sum을 검증한다. 지원 함수는 파일 전체를 1 MiB chunk로 hash한다.
- 출력: hash-verified `entries`와 allowlist DataFrame.
- 저장 Artifact: input file hashes는 최종 manifest에 저장된다.
- 검증: 16 files; 025=2,700,345, 026=1,350,162, Legacy=1,602,418.
- 실패 조건: 경로 중복/누락, byte hash mismatch, row sum mismatch.
- 다음 셀과의 관계: 검증된 entry만 streaming parser에 전달한다.

DataFrame Contract
- Grain: canonical physical ingest file 1개당 1행.
- Primary Key: `relative_path`.
- Foreign Keys: `logical_corpus` → source registry family.
- Row count expectation: 16.
- Column dictionary: role/corpus/format/path/hash/record_count.
- dtype: string 중심, record_count int64.
- nullable: 없음.
- unit: file, row.
- source: data-recon G1 ingest expectations.
- transformation: exact allowlist projection only.
- downstream consumer: streaming ingest와 artifact manifest.

In [3]:
entries = verify_ingest_allowlist(RAW_ROOT, ingest)  # 16개 raw file의 실제 bytes와 contract SHA를 독립 검증한다.
allowlist_df = pd.DataFrame(entries)  # file-grain audit table을 사람이 읽을 수 있는 DataFrame으로 만든다.
assert allowlist_df['relative_path'].is_unique  # 동일 physical file이 두 번 ingest되지 않도록 primary key를 검증한다.
assert len(allowlist_df) == 16  # canonical allowlist physical file 수를 검증한다.
display(allowlist_df[['canonical_ingest_role', 'logical_corpus', 'format', 'record_count', 'relative_path', 'sha256']])  # exact ingest scope와 hashes를 표시한다.

,canonical_ingest_role,logical_corpus,format,record_count,relative_path,sha256
0,025_EN_TO_KO_TRAIN,025,JSON,1200307,025.일상생활 및 구어체 한-영 번역 병렬 말뭉치 데이터/01.데이터/1.Trai...,922fa5e307d80436644f63ec180be387dc4b3c63b5ffaf...
1,025_KO_TO_EN_TRAIN,025,JSON,1200000,025.일상생활 및 구어체 한-영 번역 병렬 말뭉치 데이터/01.데이터/1.Trai...,5e0fdf49b5c7b3b1ca89b4e9c7b6d16c91f23b6a51e555...
2,025_EN_TO_KO_VALIDATION,025,JSON,150038,025.일상생활 및 구어체 한-영 번역 병렬 말뭉치 데이터/01.데이터/2.Vali...,ca574c2d5c14efd8513ed95a6a950ffe14896b1cfc1ada...
3,025_KO_TO_EN_VALIDATION,025,JSON,150000,025.일상생활 및 구어체 한-영 번역 병렬 말뭉치 데이터/01.데이터/2.Vali...,02dce9c0cc9c6da6ef088b410715b0397d667f349ad05d...
4,026_KO_TO_EN_TRAIN,026,JSON,1200144,026.기술과학 분야 한-영 번역 병렬 말뭉치 데이터/01.데이터/1.Trainin...,b08ea5e620cc25d23e4cd263d0146e64f1673de04d1d86...
5,026_KO_TO_EN_VALIDATION,026,JSON,150018,026.기술과학 분야 한-영 번역 병렬 말뭉치 데이터/01.데이터/2.Validat...,7e4adc920654a3321539f214ae082e273d6a38abfe59c5...
6,LEGACY_1_구어체(1),LEGACY,XLSX,200000,한국어-영어 번역(병렬) 말뭉치/1_구어체(1).xlsx,0fd712b2d62c054bed364fd41af59019a3cc6c05093d05...
7,LEGACY_1_구어체(2),LEGACY,XLSX,200000,한국어-영어 번역(병렬) 말뭉치/1_구어체(2).xlsx,f673f1c8fb73bad83e9ba097a45e47883f94d009d51ad4...
8,LEGACY_2_대화체,LEGACY,XLSX,100000,한국어-영어 번역(병렬) 말뭉치/2_대화체.xlsx,b98b105d8f5efd20d6a1a6a69071d8b13ddc337c817c17...
9,LEGACY_3_문어체_뉴스(1)_200226,LEGACY,XLSX,200011,한국어-영어 번역(병렬) 말뭉치/3_문어체_뉴스(1)/3_문어체_뉴스(1)_2002...,a4de4cb3a40831cf673be39a35b64945e1ea0838eca143...


## Cell 01.04 — Raw record streaming ingest와 provenance/content identities

- Research Spec:
  - §7.1, §11, §12.1
  - D-RD-05~07
  - Gate G1
- 목적: one structurally valid raw record→one staging row를 메모리 상한 안에서 생성한다.
- 입력: verified JSON/XLSX allowlist, research mapping.
- 전제: 025/026 `source_record_id=sn`; Legacy는 file+sheet+physical row locator; raw KO/EN은 nonempty string이다.
- 수행: ijson/openpyxl streaming, `pair_id=SHA256(length-prefix(source_id,source_record_id))`, `duplicate_group_id=SHA256(u64be(len(KO))+KO+u64be(len(EN))+EN)`. 정규화하지 않는다. `source_id`는 family+raw manifest SHA이며 official dataSetSn은 null이다.
- 출력: staging Parquet와 family row counts.
- 저장 Artifact: `data/interim/PAIR_REGISTRY_v001_staging.parquet` (재생성 가능, release 대상 아님).
- 검증: explicit Arrow schema, file별 exact record_count, KO/EN/sn/locator fail-fast.
- 실패 조건: schema drift, empty raw pair, unmapped domain, record count mismatch.
- 다음 셀과의 관계: duplicate group 전체 resolution의 입력이 된다.

DataFrame/Parquet Contract
- Grain: structurally valid raw KO-EN record 1개당 1행.
- Primary Key: 최종 검증 전 candidate `pair_id`.
- Foreign Keys: `source_id`, `raw_file_relative_path`.
- Row count expectation: 5,652,925.
- Column dictionary: D-01 raw/status fields + approved provenance fields; group-derived fields는 아직 없음.
- dtype: `staging_registry_schema()`의 explicit Arrow dtype.
- nullable: Phase1 normalization/score와 unavailable raw metadata만 nullable.
- unit: raw record.
- source: 16 verified physical files.
- transformation: identity/mapping only; text transformation 없음.
- downstream consumer: DuckDB duplicate group resolution.

In [4]:
observed_counts = write_staging_registry(RAW_ROOT, entries, research, RAW_MANIFEST_SHA256, STAGING_PATH)  # 565만 raw records를 exact allowlist 순서로 bounded-memory Parquet에 쓴다.
expected_family_counts = {key: int(ingest['expected_logical_record_counts'][key]) for key in ('025', '026', 'LEGACY')}  # independent family row oracle을 구성한다.
assert observed_counts == expected_family_counts  # raw record→row 관계가 exact인지 즉시 검증한다.
assert pq.ParquetFile(STAGING_PATH).metadata.num_rows == int(ingest['expected_logical_record_counts']['total'])  # Parquet footer row count를 독립 확인한다.
display({'staging_path': str(STAGING_PATH), 'observed_counts': observed_counts, 'staging_rows': pq.ParquetFile(STAGING_PATH).metadata.num_rows})  # streaming ingest 결과를 표시한다.

/home/sieg/projects-wsl/Tokenization_Premium/.agent_worktrees/codex-g1/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{'staging_path': '/home/sieg/projects-wsl/Tokenization_Premium/.agent_worktrees/codex-g1/data/interim/PAIR_REGISTRY_v001_staging.parquet',
 'observed_counts': {'025': 2700345, '026': 1350162, 'LEGACY': 1602418},
 'staging_rows': 5652925}

## Cell 01.05 — Duplicate-group resolution과 canonical Parquet

- Research Spec:
  - §10.1, §12.1
  - Gate G1
- 목적: content group 전체에서 representative와 conflict metadata를 계산한다.
- 입력: raw-record staging Parquet.
- 전제: representative는 provenance pointer일 뿐 semantic covariate source가 아니다.
- 수행: DuckDB out-of-core group stats; `representative_pair_id=min(pair_id)`; raw direction set이 하나면 유지, 둘 이상이면 `UNKNOWN`; domain/source conflicts는 flag만 만들고 값 선택을 발명하지 않는다. final rows는 pair_id 순으로 고정한다.
- 출력: canonical pair registry.
- 저장 Artifact: `data/registry/PAIR_REGISTRY_v001.parquet`.
- 검증: 다음 셀에서 representative/direction/group oracle을 full-table 검사한다.
- 실패 조건: DuckDB group/sort/spill 또는 Parquet persistence 실패.
- 다음 셀과의 관계: canonical table을 independent oracle과 대조한다.

Shape Contract
- symbol: R.
- physical meaning: 모든 structurally valid raw pair record registry.
- dtype: ordered Arrow table schema.
- shape: `(5,652,925, P_registry_fields)`.
- axis 0: raw provenance record.
- axis 1: D-01 및 승인/필수 engineering provenance fields.

In [5]:
finalize_registry(STAGING_PATH, PAIR_REGISTRY_PATH, RUNTIME_DIR)  # group 전체 resolution과 deterministic ordering으로 canonical Parquet을 만든다.
pair_file = pq.ParquetFile(PAIR_REGISTRY_PATH)  # 저장된 final artifact footer와 schema를 다시 연다.
assert pair_file.metadata.num_rows == int(ingest['expected_logical_record_counts']['total'])  # final artifact row count를 footer에서 검증한다.
assert pair_file.schema_arrow.names == pair_registry_schema().names  # final column order가 D-01 schema 정의와 같은지 검증한다.
display({'pair_registry_path': str(PAIR_REGISTRY_PATH), 'rows': pair_file.metadata.num_rows, 'columns': len(pair_file.schema_arrow.names), 'row_groups': pair_file.metadata.num_row_groups})  # canonical table 물리 metadata를 표시한다.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'pair_registry_path': '/home/sieg/projects-wsl/Tokenization_Premium/.agent_worktrees/codex-g1/data/registry/PAIR_REGISTRY_v001.parquet',
 'rows': 5652925,
 'columns': 44,
 'row_groups': 57}

## Cell 01.06 — Full-table identity 및 duplicate oracle reconciliation

- Research Spec:
  - §10, §12.1, §20.2
  - Gate G1
- 목적: implementation 결과를 data-recon independent oracle과 실제 final file에서 대조한다.
- 입력: canonical pair registry와 G1 expectations.
- 전제: oracle 불일치는 자동 수정 대상이 아니라 contract mismatch다.
- 수행: 100% pair_id uniqueness, exact family rows, sn collision, alias count, raw hash linkage, Phase1 null/status, 대표/방향 규칙, corpus/direction duplicate oracles를 DuckDB full scan으로 검사한다.
- 출력: machine metrics와 check-grain reconciliation rows.
- 저장 Artifact: `outputs/reports/PAIR_REGISTRY_RECONCILIATION_v001.csv`.
- 검증: 모든 row status=PASS.
- 실패 조건: 025 214252/93823, 026 44/44, cross-direction 50511, cross-corpus 0/35/1, Legacy News2↔Culture 2469 등 하나라도 불일치.
- 다음 셀과의 관계: PASS metrics가 source registry와 manifest에 연결된다.

DataFrame Contract
- Grain: acceptance check 1개당 1행.
- Primary Key: `check_id`.
- Foreign Keys: 없음.
- Row count expectation: 20개 이상, contract에 따라 고정.
- Column dictionary: check_id, observed, expected, status.
- dtype: string/int/bool 비교값.
- nullable: 없음.
- unit: check-specific.
- source: final Parquet full scans + data-recon oracle.
- transformation: exact equality 판정.
- downstream consumer: manifest validation과 Vice Director adjudication.

In [6]:
metrics, reconciliation_rows = validate_registry(PAIR_REGISTRY_PATH, ingest)  # final Parquet 전체를 독립 oracle과 fail-fast 대조한다.
write_reconciliation_csv(RECONCILIATION_PATH, reconciliation_rows)  # check-grain PASS/FAIL 표를 UTF-8 CSV로 원자 저장한다.
reconciliation_df = pd.read_csv(RECONCILIATION_PATH)  # 저장된 human report를 다시 읽어 roundtrip을 확인한다.
assert reconciliation_df['check_id'].is_unique  # reconciliation primary key uniqueness를 확인한다.
assert reconciliation_df['status'].eq('PASS').all()  # 모든 acceptance check가 실제 PASS인지 확인한다.
display(reconciliation_df)  # observed/expected/status 전체를 notebook evidence로 표시한다.

,check_id,observed,expected,status
0,total_rows,5652925,5652925,PASS
1,pair_id_unique,5652925,5652925,PASS
2,pair_id_null,0,0,PASS
3,raw_pair_structural_missing,0,0,PASS
4,phase1_status_violations,0,0,PASS
5,normalization_value_violations,0,0,PASS
6,representative_rule_violations,0,0,PASS
7,direction_resolution_violations,0,0,PASS
8,raw_hash_linkage_violations,0,0,PASS
9,canonical_physical_file_count,16,16,PASS


## Cell 01.07 — Source registry persistence

- Research Spec:
  - §9.3, §12.1, §30.2
  - D-RD-05
- 목적: record registry와 분리된 family/acquisition-grain source provenance table을 만든다.
- 입력: verified allowlist, research source portfolio, actual pair row counts.
- 전제: official AIHub dataSetSn은 미확인이므로 null이며 발명하지 않는다.
- 수행: source_id, Tier, role, primary eligibility, provenance closure, raw snapshot, license boundary, file inventory를 3행으로 저장한다.
- 출력: SOURCE_REGISTRY_v001.
- 저장 Artifact: `data/registry/SOURCE_REGISTRY_v001.parquet`.
- 검증: source_id uniqueness=100%, row count=3, expected=observed family counts.
- 실패 조건: source config 누락, count mismatch, official ID 조작.
- 다음 셀과의 관계: source artifact hash와 row count가 manifest에 기록된다.

DataFrame Contract
- Grain: local corpus family × raw manifest snapshot 1개당 1행.
- Primary Key: `source_id`.
- Foreign Keys: pair registry `source_id`.
- Row count expectation: 3.
- Column dictionary: Tier/role/eligibility/provenance/license/count/file inventory.
- dtype: `source_registry_schema()` explicit Arrow dtype.
- nullable: Legacy tier, official_dataset_id.
- unit: source snapshot.
- source: research config + ingest contract + observed rows.
- transformation: family-level aggregation only.
- downstream consumer: G1/G-ID provenance audit.

In [7]:
write_source_registry(SOURCE_REGISTRY_PATH, entries, research, RAW_MANIFEST_SHA256, metrics['row_counts'])  # 3개 family source provenance table을 explicit schema로 저장한다.
source_table = pq.read_table(SOURCE_REGISTRY_PATH)  # 저장된 source registry를 다시 읽는다.
assert source_table.num_rows == 3  # family grain row count를 검증한다.
assert source_table.schema.names == source_registry_schema().names  # source column order가 schema contract와 같은지 검증한다.
assert len(set(source_table.column('source_id').to_pylist())) == 3  # source primary key uniqueness를 검증한다.
display(source_table.to_pandas())  # source role/Tier/provenance 상태를 notebook evidence로 표시한다.

,source_id,logical_corpus,source_tier,research_role,primary_analysis_eligible,provenance_closure_status,official_dataset_id,raw_manifest_sha256,source_license_note,expected_row_count,observed_row_count,input_file_count,input_files_json,pair_version
0,025-family@raw-manifest:9a546bc91225e5331d0e8e...,025,A,PRIMARY_BACKBONE,True,PENDING_OFFICIAL_SCHEMA_AND_RELEASE_LINK,None,9a546bc91225e5331d0e8e48a1e06685cb5304ed7098af...,Raw records self-assert license=open; redistri...,2700345,2700345,4,"[{""record_count"":1200307,""relative_path"":""025....",v001
1,026-family@raw-manifest:9a546bc91225e5331d0e8e...,026,A,PRIMARY_DOMAIN_SUPPLEMENT,True,PENDING_OFFICIAL_SCHEMA_AND_RELEASE_LINK,None,9a546bc91225e5331d0e8e48a1e06685cb5304ed7098af...,Raw records self-assert license=open; redistri...,1350162,1350162,2,"[{""record_count"":1200144,""relative_path"":""026....",v001
2,Legacy-family@raw-manifest:9a546bc91225e5331d0...,LEGACY,None,SENSITIVITY_ONLY,False,PARTIAL_OFFICIAL_CONSTRUCTION_CONFIRMED_FIELD_...,None,9a546bc91225e5331d0e8e48a1e06685cb5304ed7098af...,UNKNOWN — no license metadata in source,1602418,1602418,10,"[{""record_count"":200000,""relative_path"":""한국어-영...",v001


## Cell 01.08 — Artifact manifest와 final engineering gate

- Research Spec:
  - §12.1, §30.2, §38
  - Gate G1
- 목적: pair/source/report artifacts를 input hashes, config/notebook/code commit, schema, row counts와 묶는다.
- 입력: 모든 검증 완료 artifacts와 metrics.
- 전제: 이 notebook은 ENGINEERING PASS만 보고하며 G1 최종 판정은 Vice Director 권한이다.
- 수행: 대용량 Parquet 포함 SHA-256을 계산하고 canonical UTF-8 JSON manifest를 원자 저장·재독한다.
- 출력: manifest와 compact final status.
- 저장 Artifact: `outputs/manifests/PAIR_REGISTRY_MANIFEST_v001.json`.
- 검증: artifact 존재/hash/row/schema, cross-agent contract PASS, canonical_root에 codex worktree path가 없음.
- 실패 조건: 어떤 reconciliation failure, artifact/hash 누락, agent worktree canonical_root leak.
- 다음 셀과의 관계: 없음; G1 integration candidate handoff다.

In [8]:
manifest = build_manifest(  # contracts/code/input/output lineage와 SHA-256을 하나의 manifest로 구성한다.
    execution_root=EXECUTION_ROOT,  # artifact 상대경로와 Git commit을 실제 실행 worktree에서 계산한다.
    canonical_root=CANONICAL_ROOT,  # manifest canonical identity는 agent worktree가 아닌 project root로 고정한다.
    raw_root=RAW_ROOT,  # immutable raw bytes 위치를 lineage에 기록한다.
    research_config_path=RESEARCH_CONFIG_PATH,  # 실제 소비한 research contract path/hash를 연결한다.
    ingest_contract_path=INGEST_CONTRACT_PATH,  # 실제 소비한 data-recon oracle path/hash를 연결한다.
    notebook_path=NOTEBOOK_PATH,  # fresh-kernel 입력 notebook hash를 연결한다.
    pair_registry_path=PAIR_REGISTRY_PATH,  # canonical pair artifact path/hash/rows를 연결한다.
    source_registry_path=SOURCE_REGISTRY_PATH,  # source artifact path/hash/rows를 연결한다.
    reconciliation_path=RECONCILIATION_PATH,  # human validation report path/hash/rows를 연결한다.
    entries=entries,  # 16개 verified input file inventory를 전달한다.
    raw_manifest_sha256=RAW_MANIFEST_SHA256,  # source snapshot identity hash를 전달한다.
    metrics=metrics,  # full-table oracle metrics를 전달한다.
)  # 완성된 machine-readable manifest payload를 받는다.
manifest['cross_agent_contract'] = 'PASS'  # 실제 Claude/data contract load와 full oracle PASS를 machine-readable gate로 기록한다.
persist_manifest(MANIFEST_PATH, manifest)  # canonical UTF-8 JSON으로 저장하고 roundtrip/canonical-root leak를 검증한다.
assert manifest['pair_registry']['row_count'] == 5_652_925  # final pair artifact total rows를 다시 확인한다.
assert manifest['source_registry']['row_count'] == 3  # final source artifact rows를 다시 확인한다.
assert manifest['validation']['status'] == 'PASS'  # full-table oracle status를 다시 확인한다.
assert manifest['cross_agent_contract'] == 'PASS'  # cross-agent contract gate를 다시 확인한다.
assert '.agent_worktrees/codex' not in manifest['canonical_root']  # canonical_root가 agent worktree를 가리키지 않는지 최종 확인한다.
final_status = {  # 과장 없는 final engineering status 표를 구성한다.
    'engineering_verdict': 'PASS',  # Codex 권한 범위의 engineering 결과만 PASS로 표시한다.
    'g1_adjudication': 'VICE_DIRECTOR_REQUIRED',  # 연구 Gate 최종 판정 권한을 명시한다.
    'pair_rows': manifest['pair_registry']['row_count'],  # canonical pair row count를 표시한다.
    'pair_sha256': manifest['pair_registry']['sha256'],  # canonical pair bytes hash를 표시한다.
    'source_rows': manifest['source_registry']['row_count'],  # source registry row count를 표시한다.
    'source_sha256': manifest['source_registry']['sha256'],  # source registry bytes hash를 표시한다.
    'manifest_path': str(MANIFEST_PATH),  # persisted manifest 경로를 표시한다.
}  # final status mapping 구성을 마친다.
display(final_status)  # final engineering status를 notebook evidence로 표시한다.

{'engineering_verdict': 'PASS',
 'g1_adjudication': 'VICE_DIRECTOR_REQUIRED',
 'pair_rows': 5652925,
 'pair_sha256': 'eac9f4cc37d4a394d81d17c24b7910716c3bc511dbc15dd3929be1cb9393e2a4',
 'source_rows': 3,
 'source_sha256': '4c359c072a8f27412b52db6fe334e25ca8e2a42b46c742ce1fadbf84e1b09a94',
 'manifest_path': '/home/sieg/projects-wsl/Tokenization_Premium/.agent_worktrees/codex-g1/outputs/manifests/PAIR_REGISTRY_MANIFEST_v001.json'}